In [4]:
import pandas as pd

# 1. Define your paths and tickers
# Adjust the paths below to match your actual folder structure
files = {
    'AAPL': '../data/raw/AAPL.csv',
    'AMZN': '../data/raw/AMZN.csv',
    'GOOG': '../data/raw/GOOG.csv',
    'META': '../data/raw/META.csv',
    'NVDA': '../data/raw/NVDA.csv'
}

stock_data = {}

# 2. Process each file
for ticker, path in files.items():
    print(f"Processing {ticker}...")
    
    # Load data
    df = pd.read_csv(path)
    
    # Standardize Date column
    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index('Date', inplace=True)
    df.sort_index(inplace=True)
    
    # Ensure numeric types for OHLCV
    cols = ['Open', 'High', 'Low', 'Close', 'Volume']
    for col in cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Handle missing values (Interpolation is best for financial time-series)
    if df.isnull().values.any():
        print(f"  - Fixed missing values in {ticker}")
        df = df.interpolate(method='time').ffill().bfill()
        
    stock_data[ticker] = df

print("\nAll datasets loaded and cleaned.")


# Check the first few rows of one stock and the overall info
display(stock_data['AAPL'].head())
stock_data['AAPL'].info()

Processing AAPL...
Processing AMZN...
Processing GOOG...
Processing META...
Processing NVDA...

All datasets loaded and cleaned.


,Close,High,Low,Open,Volume
Date,,,,,
2009-01-02,2.721686,2.730385,2.554037,2.575630,746015200
2009-01-05,2.836553,2.884539,2.780469,2.794266,1181608400
2009-01-06,2.789767,2.914229,2.770872,2.877641,1289310400
2009-01-07,2.729484,2.774170,2.706990,2.753477,753048800
2009-01-08,2.780169,2.793666,2.700393,2.712090,673500800


<class 'pandas.DataFrame'>
DatetimeIndex: 3774 entries, 2009-01-02 to 2023-12-29
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   3774 non-null   float64
 1   High    3774 non-null   float64
 2   Low     3774 non-null   float64
 3   Open    3774 non-null   float64
 4   Volume  3774 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 176.9 KB


In [ ]:
import talib

for ticker in stock_data:
    df = stock_data[ticker]
    
    # 1. Moving Averages (SMA & EMA)
    # We'll use a 20-day window for short-term and 50-day for medium-term
    df['SMA_20'] = talib.SMA(df['Close'], timeperiod=20)
    df['SMA_50'] = talib.SMA(df['Close'], timeperiod=50)
    df['EMA_20'] = talib.EMA(df['Close'], timeperiod=20)
    
    # 2. RSI (Relative Strength Index)
    # Standard 14-day window to identify overbought (>70) or oversold (<30)
    df['RSI'] = talib.RSI(df['Close'], timeperiod=14)
    
    # 3. MACD (Moving Average Convergence Divergence)
    # Standard parameters: Fast=12, Slow=26, Signal=9
    macd, macdsignal, macdhist = talib.MACD(df['Close'], 
                                            fastperiod=12, 
                                            slowperiod=26, 
                                            signalperiod=9)
    df['MACD'] = macd
    df['MACD_Signal'] = macdsignal
    df['MACD_Hist'] = macdhist
    
    # Save the updated DataFrame back to the dictionary
    stock_data[ticker] = df

print(f"Technical indicators computed for: {', '.join(stock_data.keys())}")